In [4]:
import numpy as np


def multi_head_attention(input_embeddings, query_projection_weights,
                         key_projection_weights, value_projection_weights,
                         output_projection_weights, number_of_heads):
    """
    Return output (seq_len, d_model) and weights
    (num_heads, seq_len, seq_len).
    """

    query = np.matmul(input_embeddings, query_projection_weights)
    key = np.matmul(input_embeddings, key_projection_weights)
    value = np.matmul(input_embeddings, value_projection_weights)

    sequence_length, model_dimension = input_embeddings.shape
    head_dimension = model_dimension // number_of_heads

    assert model_dimension % number_of_heads == 0, "d_model must be divisible by num_heads"

    query = query.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    key = key.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)
    value = value.reshape(sequence_length, number_of_heads, head_dimension).transpose(1, 0, 2)

    attention_scores = np.matmul(query, key.transpose(0, 2, 1)) / np.sqrt(head_dimension)

    attention_scores = attention_scores - np.max(attention_scores, axis=-1, keepdims=True)
    exponentiated_scores = np.exp(attention_scores)

    attention_weights = exponentiated_scores / np.sum(exponentiated_scores, axis=-1, keepdims=True)

    per_head_outputs = np.matmul(attention_weights, value)

    concatenated_heads = per_head_outputs.transpose(1, 0, 2).reshape(sequence_length, model_dimension)

    attention_output = np.matmul(concatenated_heads, output_projection_weights)

    return attention_output, attention_weights

def layer_norm(input_embeddings, layer_norm_scale, layer_norm_shift, epsilon=1e-5):

    feature_mean = np.mean(input_embeddings, axis=-1, keepdims=True)
    feature_variance = np.var(input_embeddings, axis=-1, keepdims=True)

    normalized_embeddings = (input_embeddings - feature_mean) / np.sqrt(feature_variance + epsilon)

    return layer_norm_scale * normalized_embeddings + layer_norm_shift

def feed_forward(input_embeddings, feed_forward_weights_1, feed_forward_bias_1,
                 feed_forward_weights_2, feed_forward_bias_2):
    hidden_activations = np.maximum(
        0, input_embeddings @ feed_forward_weights_1 + feed_forward_bias_1
    )  # ReLU
    feed_forward_output = hidden_activations @ feed_forward_weights_2 + feed_forward_bias_2
    return feed_forward_output

def transformer_encoder_block(input_embeddings, model_parameters):
    """Return (seq_len, d_model) output."""

    # Multi-Head Self-Attention (reuse previous function)
    attention_output, attention_weights = multi_head_attention(
        input_embeddings,
        model_parameters['query_projection_weights'],
        model_parameters['key_projection_weights'],
        model_parameters['value_projection_weights'],
        model_parameters['output_projection_weights'],
        model_parameters['number_of_heads']
    )

    # Residual Connection + Layer Normalization
    intermediate_embeddings = layer_norm(
        input_embeddings + attention_output,
        model_parameters['layer_norm_scale_1'],
        model_parameters['layer_norm_shift_1']
    )

    # Feed Forward Network
    feed_forward_output = feed_forward(
        intermediate_embeddings,
        model_parameters['feed_forward_weights_1'],
        model_parameters['feed_forward_bias_1'],
        model_parameters['feed_forward_weights_2'],
        model_parameters['feed_forward_bias_2']
    )

    # Residual Connection + Layer Normalization
    encoder_output = layer_norm(
        intermediate_embeddings + feed_forward_output,
        model_parameters['layer_norm_scale_2'],
        model_parameters['layer_norm_shift_2']
    )

    return encoder_output

In [2]:
sequence_length = 5
model_dimension = 8
number_of_heads = 2
feed_forward_dimension = 16

sample_input_embeddings = np.random.randn(sequence_length, model_dimension)

In [5]:
model_parameters = {
    'query_projection_weights': np.random.randn(model_dimension, model_dimension) * 0.1,
    'key_projection_weights': np.random.randn(model_dimension, model_dimension) * 0.1,
    'value_projection_weights': np.random.randn(model_dimension, model_dimension) * 0.1,
    'output_projection_weights': np.random.randn(model_dimension, model_dimension) * 0.1,

    'feed_forward_weights_1': np.random.randn(model_dimension, feed_forward_dimension) * 0.1,
    'feed_forward_bias_1': np.zeros(feed_forward_dimension),

    'feed_forward_weights_2': np.random.randn(feed_forward_dimension, model_dimension) * 0.1,
    'feed_forward_bias_2': np.zeros(model_dimension),

    'layer_norm_scale_1': np.ones(model_dimension),
    'layer_norm_shift_1': np.zeros(model_dimension),

    'layer_norm_scale_2': np.ones(model_dimension),
    'layer_norm_shift_2': np.zeros(model_dimension),

    'number_of_heads': number_of_heads
}

transformer_encoder_output = transformer_encoder_block(
    sample_input_embeddings, model_parameters
)

print("Input Shape:", sample_input_embeddings.shape)
print("Output Shape:", transformer_encoder_output.shape)
print("\nEncoder Output:\n", transformer_encoder_output)


Input Shape: (5, 8)
Output Shape: (5, 8)

Encoder Output:
 [[-2.30043381 -0.41689112  0.9913696   0.32014145  1.07806841  0.2589442
   0.36392251 -0.29512124]
 [-0.44112192 -1.00494419 -0.41696381  0.88041057  1.14964899 -0.55579775
  -1.24441179  1.63317992]
 [-0.23527573  1.70113797 -1.38043123 -1.11763739  0.92150555  0.20601571
   0.65938907 -0.75470395]
 [-1.51202185  0.39994145  0.60371868 -0.04588469  0.30205629 -1.23765063
   1.85269696 -0.36285622]
 [ 1.59137437  1.03116237 -0.8083426  -0.83739547  0.70067218  0.14890456
  -0.2541857  -1.57218971]]
